# 06_coordinator

06_coordinator.py — 동적 라우팅: LLM 이 sub_agent 에 위임

워크플로우 에이전트가 *고정 순서* 라면, LlmAgent 에 sub_agents 를 붙이면
*LLM 이 판단해서* 적절한 하위 에이전트로 위임. supervisor/dispatcher 패턴.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 이미 이벤트 루프가 돌아 스크립트의 asyncio.run() 이 깨짐 → nest_asyncio 로 중첩 허용
import nest_asyncio; nest_asyncio.apply()
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '06_coordinator.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
06_coordinator.py — 동적 라우팅: LLM 이 sub_agent 에 위임

워크플로우 에이전트가 *고정 순서* 라면, LlmAgent 에 sub_agents 를 붙이면
*LLM 이 판단해서* 적절한 하위 에이전트로 위임. supervisor/dispatcher 패턴.
"""
from google.adk.agents import LlmAgent

from _adk_common import adk_model, run_once, banner, adk_unavailable


def main() -> None:
    banner("Coordinator — LLM 기반 동적 위임 (transfer)")

    billing = LlmAgent(
        name="billing",
        model=adk_model(),
        description="결제·환불 문의를 처리한다.",
        instruction=(
            "너는 결제팀 담당이다. 결제 / 환불 / 청구 관련 문의에만 답하라. "
            "한국어 한 문장."
        ),
    )

    support = LlmAgent(
        name="support",
        model=adk_model(),
        description="기술 지원 문의를 처리한다.",
        instruction=(
            "너는 기술지원팀 담당이다. 제품 사용법 / 에러 / 버그 문의에만 답하라. "
            "한국어 한 문장."
        ),
    )

    coordinator = LlmAgent(
        name="coordinator",
        model=adk_model(),
        description="고객 문의 라우터",
        instruction=(
            "사용자 문의를 분석해서 결제 관련이면 billing, 기술 문의면 support "
            "에게 transfer 하라. 직접 답하지 말고 반드시 위임하라."
        ),
        sub_agents=[billing, support],
    )

    cases = [
        "환불 신청 방법 알려줘",
        "앱이 자꾸 강제종료돼요",
    ]
    for q in cases:
        print(f"\n  ❓ {q}")
        try:
            reply = run_once(coordinator, q)
            print(f"  💬 {reply[:200]}")
        except Exception as e:
            print(f"  ⚠ {type(e).__name__}: {str(e)[:120]}")
            adk_unavailable()
            break


if __name__ == "__main__":
    main()


📌 Coordinator — LLM 기반 동적 위임 (transfer)

  ❓ 환불 신청 방법 알려줘


D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\google\adk\tools\function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



Node execution failed with exception
Traceback (most recent call last):
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\llm_http_handler.py", line 180, in _make_common_async_call
    response = await async_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 574, in post
    await _raise_masked_async_error(e, stream)
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 363, in _raise_masked_async_error
    raise MaskedHTTPStatusError(e, message=_text, text=_text) from None
litellm.llms.custom_httpx.http_handler.MaskedHTTPStatusError: Cl

Root node coordinator failed.
Traceback (most recent call last):
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\llm_http_handler.py", line 180, in _make_common_async_call
    response = await async_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 574, in post
    await _raise_masked_async_error(e, stream)
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 363, in _raise_masked_async_error
    raise MaskedHTTPStatusError(e, message=_text, text=_text) from None
litellm.llms.custom_httpx.http_handler.MaskedHTTPStatusError: Client er


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

  ⚠ RateLimitError: litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Provider returned error","code":429,"
⚠️ ADK 실행 실패 — google-adk[extensions] 설치 + OPENROUTER_API_KEY 확인.
